# Decoder UI - interactive 3D reconstruction tester

A small **Gradio** app to test the trained decoders. Pick a model (**U-Net** or **V-Net**) and a
**checkpoint epoch**, choose a built-in test case *or upload your own AP + LAT images* (e.g. real
**Regen** clinical X-rays), and view the reconstructed 3D bone surface plus mid-slices. When a
ground-truth CT exists for the chosen case it also reports Dice/IoU.

This is how you **revisit any saved epoch** and how doctors can **qualitatively review** Regen
reconstructions (which have no 3D ground truth).

### Setup

This app needs **gradio** (not required for training). Install it once, then restart the kernel:

In [2]:
import os, tempfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import timm
import nibabel as nib
from skimage import measure
import matplotlib.pyplot as plt
try:
    import gradio as gr
    HAS_GRADIO = True
except Exception:
    HAS_GRADIO = False
    print("Gradio not installed -> run the install cell above, then restart the kernel.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", DEVICE, "| gradio available:", HAS_GRADIO)

device cpu | gradio available: True


In [3]:
def find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "interim" / "predrr").exists():
            return cand
    raise FileNotFoundError("project root not found (expected data/interim/predrr)")

ROOT           = find_root(Path.cwd())
MODELS_DIR     = ROOT / "models"
DEC_DIR        = MODELS_DIR / "decoders"
PREDRR_DIR     = ROOT / "data" / "interim" / "predrr"
NORMAL_DRR_DIR = ROOT / "data" / "interim" / "DRRs"
# encoder defaults (weights come from the checkpoint, so no ImageNet fetch / no fine-tuning here)
PRETRAINED     = False
FREEZE_ENCODER = True
print("ROOT", ROOT)

ROOT C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject


### Shared encoder (copied verbatim from `encoder_pipeline.ipynb`)

The encoder is the part both decoders share, so the comparison is fair: **only the decoder
changes**. The code below is copied verbatim from `encoder_pipeline.ipynb` so this notebook is
self-contained — **do not edit it here**. (When `encoder_pipeline.ipynb` is later converted to a
`.py` module, replace this cell with a simple `import`.)

What it does, in plain terms:
1. A **ConvNeXtV2** backbone turns each X-ray (AP and LAT) into 4 feature maps at increasing depth.
2. **Hybrid bi-planar fusion** merges the two views: cheap convolution at fine scales (keeps local
   fracture detail), cross-attention at coarse scales (aligns global knee shape).
3. A **2D->3D lift** stacks each fused map into a small 3D feature volume (depth = `LIFT_DEPTH`).

Output: a list of 4 multi-scale 3D feature tensors with channels `[64, 128, 256, 512]` — this is
the *contract* the decoder consumes.

In [4]:
# ===== Encoder front-end - VERBATIM from encoder_pipeline.ipynb. DO NOT EDIT. =====
# (PRETRAINED / FREEZE_ENCODER are set in the CONFIG cell so they stay visible knobs.)
BACKBONE     = "convnextv2_tiny"
IMG_SIZE     = 256
OUT_CHANNELS = [64, 128, 256, 512]
FUSION_TYPES = ["local", "local", "attn", "attn"]   # fine -> coarse

def make_backbone(pretrained=True):
    """features_only ConvNeXtV2 returning 4 multi-scale maps. Falls back to random init offline."""
    try:
        return timm.create_model(BACKBONE, pretrained=pretrained, features_only=True)
    except Exception as e:
        print("[warn] pretrained fetch failed (%s); random init." % type(e).__name__)
        return timm.create_model(BACKBONE, pretrained=False, features_only=True)

FEAT_DIMS = [f["num_chs"] for f in make_backbone(pretrained=False).feature_info]   # [96,192,384,768]

def load_drr(path):
    """npy 256x256 float32 [0,1] -> tensor [3,H,W] (1 channel replicated to 3 for ConvNeXtV2)."""
    arr = np.load(path).astype(np.float32)
    t = torch.from_numpy(arr)
    if t.ndim == 2:
        t = t.unsqueeze(0)
    return t.repeat(3, 1, 1) if t.shape[0] == 1 else t

NORMALIZE = T.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
def paired_tf(t):
    return NORMALIZE(t)

class CrossAttention(nn.Module):
    """AP (query) attends to LAT (key/value). Operates on tokens [B, N, C]."""
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim); self.k = nn.Linear(dim, dim); self.v = nn.Linear(dim, dim)
        self.scale = dim ** -0.5
    def forward(self, a, b):
        attn = F.softmax(torch.matmul(self.q(a), self.k(b).transpose(-2, -1)) * self.scale, dim=-1)
        return torch.matmul(attn, self.v(b)) + a

class LocalFusion(nn.Module):
    """Cheap high-res fusion: concat views + 3x3 conv, residual on AP."""
    def __init__(self, dim):
        super().__init__()
        self.mix = nn.Conv2d(2 * dim, dim, kernel_size=3, padding=1)
    def forward(self, a, b):
        return self.mix(torch.cat([a, b], dim=1)) + a

class BiPlanarFeatureFusion(nn.Module):
    def __init__(self, feat_dims=FEAT_DIMS, out_channels=OUT_CHANNELS,
                 fusion_types=FUSION_TYPES, depth=16, pretrained=True, freeze_encoder=False):
        super().__init__()
        self.encoder = make_backbone(pretrained)
        self.fusion_types = list(fusion_types); self.depth = depth
        self.fuse = nn.ModuleList([CrossAttention(d) if t == "attn" else LocalFusion(d)
                                   for d, t in zip(feat_dims, fusion_types)])
        self.to3d = nn.ModuleList([nn.Conv2d(c, o, 1) for c, o in zip(feat_dims, out_channels)])
        self.expand3d = nn.ModuleList([nn.Conv3d(o, o, 3, padding=1) for o in out_channels])
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
    def load_simclr_encoder(self, path):
        missing, unexpected = self.encoder.load_state_dict(torch.load(path, map_location="cpu"), strict=False)
        print("loaded SimCLR encoder: missing=%d unexpected=%d" % (len(missing), len(unexpected)))
    def forward(self, ap_img, lat_img):
        ap_feats, lat_feats = self.encoder(ap_img), self.encoder(lat_img)
        fused2d, fused3d = [], []
        for ap_f, lat_f, fuse, c2d, c3d, t in zip(
                ap_feats, lat_feats, self.fuse, self.to3d, self.expand3d, self.fusion_types):
            B, C, H, W = ap_f.shape
            if t == "attn":
                a = ap_f.flatten(2).transpose(1, 2); b = lat_f.flatten(2).transpose(1, 2)
                f2d = fuse(a, b).transpose(1, 2).reshape(B, C, H, W)
            else:
                f2d = fuse(ap_f, lat_f)
            fused2d.append(f2d)
            f3 = c2d(f2d).unsqueeze(2)
            f3 = F.interpolate(f3, size=(self.depth, H, W), mode="trilinear", align_corners=False)
            fused3d.append(c3d(f3))
        return fused2d, fused3d

print("encoder feature dims:", FEAT_DIMS)

encoder feature dims: [96, 192, 384, 768]


In [5]:
def conv_block(block_type, in_ch, out_ch):
    return DoubleConv(in_ch, out_ch) if block_type == "unet" else VNetResBlock(in_ch, out_ch)

class DoubleConv(nn.Module):
    """U-Net block: (Conv3d -> BN -> ReLU) x2. Plain, no residual."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)

class VNetResBlock(nn.Module):
    """V-Net block: (Conv3d -> BN -> PReLU) x2 + residual add (input projected if channels differ)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.proj = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.c1 = nn.Conv3d(in_ch, out_ch, 3, padding=1); self.n1 = nn.BatchNorm3d(out_ch); self.a1 = nn.PReLU(out_ch)
        self.c2 = nn.Conv3d(out_ch, out_ch, 3, padding=1); self.n2 = nn.BatchNorm3d(out_ch); self.a2 = nn.PReLU(out_ch)
    def forward(self, x):
        y = self.a1(self.n1(self.c1(x)))
        y = self.n2(self.c2(y))
        return self.a2(y + self.proj(x))

class SuperResHead(nn.Module):
    """Grow the (LIFT_DEPTH, 64, 64) feature grid up to (T,T,T) via staged trilinear upsample +
    refine blocks with tapering channels (heavy work stays at low resolution -> low memory)."""
    def __init__(self, block_type, in_ch, target, use_grad_ckpt=False):
        super().__init__()
        self.target = tuple(int(t) for t in target); self.use_grad_ckpt = use_grad_ckpt
        self.b1 = conv_block(block_type, in_ch, 32)
        self.b2 = conv_block(block_type, 32, 16)
        self.b3 = conv_block(block_type, 16, 8)
        self.out = nn.Conv3d(8, 1, 1)
    def _run(self, blk, x):
        if self.use_grad_ckpt and x.requires_grad:
            return cp.checkpoint(blk, x, use_reentrant=False)
        return blk(x)
    def forward(self, x):
        d0, h0, w0 = x.shape[-3:]; dt, ht, wt = self.target
        s1 = (round(d0 + (dt - d0) / 3), round(h0 + (ht - h0) / 3), round(w0 + (wt - w0) / 3))
        s2 = (round(d0 + 2 * (dt - d0) / 3), round(h0 + 2 * (ht - h0) / 3), round(w0 + 2 * (wt - w0) / 3))
        x = F.interpolate(x, size=s1, mode="trilinear", align_corners=False); x = self._run(self.b1, x)
        x = F.interpolate(x, size=s2, mode="trilinear", align_corners=False); x = self._run(self.b2, x)
        x = F.interpolate(x, size=self.target, mode="trilinear", align_corners=False); x = self._run(self.b3, x)
        return self.out(x)

class Decoder3D(nn.Module):
    """Multi-scale skip-connected decoder. Same wiring for both models; only the block differs."""
    def __init__(self, block_type, enc_channels=OUT_CHANNELS, target=(64, 64, 64),
                 use_grad_ckpt=False, deep_supervision=False):
        super().__init__()
        c0, c1, c2, c3 = enc_channels
        self.deep_supervision = deep_supervision; self.target = tuple(int(t) for t in target)
        self.up3 = nn.ConvTranspose3d(c3, c2, kernel_size=(1, 2, 2), stride=(1, 2, 2))
        self.dec3 = conv_block(block_type, c2 + c2, c2)
        self.up2 = nn.ConvTranspose3d(c2, c1, kernel_size=(1, 2, 2), stride=(1, 2, 2))
        self.dec2 = conv_block(block_type, c1 + c1, c1)
        self.up1 = nn.ConvTranspose3d(c1, c0, kernel_size=(1, 2, 2), stride=(1, 2, 2))
        self.dec1 = conv_block(block_type, c0 + c0, c0)
        self.sr = SuperResHead(block_type, c0, self.target, use_grad_ckpt)
        if deep_supervision:
            self.aux3 = nn.Conv3d(c2, 1, 1); self.aux2 = nn.Conv3d(c1, 1, 1); self.aux1 = nn.Conv3d(c0, 1, 1)
    def forward(self, feats):
        l0, l1, l2, l3 = feats
        x = self.up3(l3); x = torch.cat([x, l2], 1); x = self.dec3(x); a3 = x
        x = self.up2(x);  x = torch.cat([x, l1], 1); x = self.dec2(x); a2 = x
        x = self.up1(x);  x = torch.cat([x, l0], 1); x = self.dec1(x); a1 = x
        out = self.sr(x)
        if self.deep_supervision and self.training:
            up = lambda h: F.interpolate(h, size=self.target, mode="trilinear", align_corners=False)
            return out, [up(self.aux3(a3)), up(self.aux2(a2)), up(self.aux1(a1))]
        return out, None

class ReconModel(nn.Module):
    """Full model = shared bi-planar encoder/fusion + a (U-Net or V-Net) decoder."""
    def __init__(self, fusion, decoder):
        super().__init__(); self.fusion = fusion; self.decoder = decoder
    def forward(self, ap, lat):
        _, f3d = self.fusion(ap, lat)
        return self.decoder(f3d)

### Inference helpers

`load_model_from_ckpt` rebuilds the exact architecture recorded in the checkpoint's config
(model type, resolution, lift depth) and loads its weights. `to_input` normalises any image to the
encoder's expected `3x256x256` tensor. `volume_to_obj` turns the predicted occupancy volume into a
surface mesh with marching cubes and writes a small `.obj` for the 3D viewer.

In [6]:
MODEL_CACHE = {"path": None, "model": None, "cfg": None}

def list_checkpoints(model_type):
    d = DEC_DIR / model_type
    return sorted(p.name for p in d.glob("%s_*.pth" % model_type)) if d.exists() else []

def load_model_from_ckpt(model_type, ckpt_name):
    path = DEC_DIR / model_type / ckpt_name
    if MODEL_CACHE["path"] == str(path):
        return MODEL_CACHE["model"], MODEL_CACHE["cfg"]
    ck = torch.load(str(path), map_location=DEVICE)
    cfg = ck.get("config", {"MODEL": model_type, "TARGET_RES": 256, "LIFT_DEPTH": 16, "GT_THRESH": 0.4})
    fusion = BiPlanarFeatureFusion(depth=cfg.get("LIFT_DEPTH", 16), pretrained=False, freeze_encoder=True)
    decoder = Decoder3D(cfg.get("MODEL", model_type), target=(cfg.get("TARGET_RES", 256),) * 3)
    model = ReconModel(fusion, decoder).to(DEVICE).eval()
    model.load_state_dict(ck["model"])
    MODEL_CACHE.update(path=str(path), model=model, cfg=cfg)
    return model, cfg

def to_input(img):
    """Any HxW or HxWxC array -> normalised [3,256,256] tensor."""
    a = np.asarray(img).astype(np.float32)
    if a.ndim == 3:
        a = a.mean(-1)
    a = a - a.min(); a = a / (a.max() + 1e-8)
    t = F.interpolate(torch.from_numpy(a)[None, None], size=(256, 256), mode="bilinear", align_corners=False)[0]
    return paired_tf(t.repeat(3, 1, 1))

def save_obj(path, verts, faces):
    with open(path, "w") as f:
        for v in verts:
            f.write("v %f %f %f\n" % (v[0], v[1], v[2]))
        for t in faces:
            f.write("f %d %d %d\n" % (t[0] + 1, t[1] + 1, t[2] + 1))

def volume_to_obj(vol, thr=0.5):
    # marching_cubes needs the level strictly inside the value range, and enough surface to mesh
    vmin, vmax = float(vol.min()), float(vol.max())
    if int((vol > thr).sum()) < 10 or not (vmin < thr < vmax):
        return None
    try:
        verts, faces, _, _ = measure.marching_cubes(vol, level=thr)
    except (ValueError, RuntimeError):
        return None
    out = Path(tempfile.gettempdir()) / "recon_mesh.obj"
    save_obj(out, verts, faces)
    return str(out)

def list_test_cases():
    cases = []
    for ds in ["healthy", "fractured"]:
        d = NORMAL_DRR_DIR / ds
        if not d.exists():
            continue
        for case in sorted(os.listdir(d)):
            for side in ["left", "right"]:
                if (d / case / side / "ap.npy").exists() and (d / case / side / "lat.npy").exists():
                    cases.append("%s/%s/%s" % (ds, case, side))
    return cases

def gt_for(label):
    ds, case, side = label.split("/")
    Side = "Right" if side.startswith("r") else "Left"
    p = (PREDRR_DIR / "healthy" / ("%s_%s.nii.gz" % (case, Side)) if ds == "healthy"
         else PREDRR_DIR / "fractured" / ("%s_Part%s.nii.gz" % (case, Side)))
    return p if p.exists() else None

### The app

Press **Reconstruct** to run inference. The 3D viewer shows the predicted bone surface; the plot
shows the input AP view and two prediction slices; the info box reports the checkpoint used and, for
built-in cases with a CT, the Dice/IoU.

In [7]:
def run(model_type, ckpt_name, source, case_label, ap_up, lat_up, thr):
    if not ckpt_name:
        return None, None, "Pick a checkpoint first (train a model, or check models/decoders/)."
    model, cfg = load_model_from_ckpt(model_type, ckpt_name)
    if source == "test case":
        ds, case, side = case_label.split("/")
        ap_arr = np.load(NORMAL_DRR_DIR / ds / case / side / "ap.npy")
        lat_arr = np.load(NORMAL_DRR_DIR / ds / case / side / "lat.npy")
    else:
        if ap_up is None or lat_up is None:
            return None, None, "Upload both an AP and a LAT image."
        ap_arr, lat_arr = ap_up, lat_up
    ap = to_input(ap_arr).unsqueeze(0).to(DEVICE); lat = to_input(lat_arr).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out, _ = model(ap, lat)
    vol = torch.sigmoid(out[0, 0]).cpu().numpy()
    obj = volume_to_obj(vol, thr)

    disp = np.asarray(ap_arr); disp = disp.mean(-1) if disp.ndim == 3 else disp
    fig, ax = plt.subplots(1, 3, figsize=(10, 3.3))
    ax[0].imshow(disp, cmap="gray"); ax[0].set_title("input AP")
    ax[1].imshow(vol[vol.shape[0] // 2] > thr, cmap="gray"); ax[1].set_title("axial mid")
    ax[2].imshow(vol[:, vol.shape[1] // 2, :] > thr, cmap="gray"); ax[2].set_title("coronal mid")
    for a in ax:
        a.axis("off")
    plt.tight_layout()

    msg = "model=%s | ckpt=%s | TARGET_RES=%s" % (model_type, ckpt_name, cfg.get("TARGET_RES"))
    if source == "test case":
        gp = gt_for(case_label)
        if gp is not None:
            gv = nib.load(str(gp)).get_fdata().astype(np.float32)
            gt = F.interpolate(torch.from_numpy((gv > cfg.get("GT_THRESH", 0.4)).astype(np.float32))[None, None],
                               size=vol.shape, mode="nearest")[0, 0].numpy()
            p = (vol > thr).astype(np.float32); inter = float((p * gt).sum())
            dice = 2 * inter / (p.sum() + gt.sum() + 1e-6)
            iou = inter / (p.sum() + gt.sum() - inter + 1e-6)
            msg += " | Dice=%.3f IoU=%.3f" % (dice, iou)
        else:
            msg += " | (no ground-truth CT for this case)"
    if obj is None:
        msg += " | [empty reconstruction - try a lower threshold or a more-trained checkpoint]"
    return obj, fig, msg

def build_ui():
    with gr.Blocks(title="Knee 3D Reconstruction") as demo:
        gr.Markdown("# Knee 3D Reconstruction - U-Net vs V-Net")
        with gr.Row():
            model_type = gr.Dropdown(["unet", "vnet"], value="unet", label="Model")
            ckpt = gr.Dropdown(list_checkpoints("unet"), label="Checkpoint (epoch)")
        model_type.change(lambda mt: gr.update(choices=list_checkpoints(mt),
                                               value=(list_checkpoints(mt) or [None])[0]),
                          model_type, ckpt)
        source = gr.Radio(["test case", "upload"], value="test case", label="Input source")
        case = gr.Dropdown(list_test_cases(), label="Built-in test case (dataset/case/side)")
        with gr.Row():
            ap_up = gr.Image(label="AP X-ray (upload)", type="numpy")
            lat_up = gr.Image(label="LAT X-ray (upload)", type="numpy")
        thr = gr.Slider(0.1, 0.9, value=0.5, step=0.05, label="Occupancy threshold")
        btn = gr.Button("Reconstruct", variant="primary")
        with gr.Row():
            mesh = gr.Model3D(label="3D reconstruction")
            slices = gr.Plot(label="Slices")
        info = gr.Textbox(label="Info / metrics")
        btn.click(run, [model_type, ckpt, source, case, ap_up, lat_up, thr], [mesh, slices, info])
    return demo

if HAS_GRADIO:
    demo = build_ui()
    demo.launch()
else:
    print("Install gradio (cell at the top), restart the kernel, then re-run.")

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Notes

- The dropdown lists checkpoints from `models/decoders/<model>/`. Train at least one model first.
- For **Regen** clinical X-rays, choose **upload** and provide the AP and LAT images; there is no
  ground truth, so you get the mesh + slices for qualitative review (no Dice/IoU).
- `demo.launch(share=True)` gives a temporary public link if you need to show it to a supervisor.